In [ ]:
# standaloneFL_GAT.py  (new file at project root)
import sys, os, pickle, random
import numpy as np
import flwr as fl
import torch as t
from collections import OrderedDict
from typing import List, Tuple
from sklearn.model_selection import StratifiedKFold

from sources.iongreen2_analysisPaper import scanGenes, buildFeatVect, buildVectorGeneWise, checkVectors
from sources.readPhenopedia import readPhenopedia
from sources import utils as U
from sources.GATModel import GATCrohnModel
from sources.buildGeneGraph import build_adj_from_string, build_adj_phenopedia
from sources.FedExplainer import LocalExplainer, GlobalExplainer
from sources.PersonalizedFL import PersonalizedGATCrohn
import sources.Constants as CONST

MIN_REFERENCE_NUM = 0
DBs = [
    "marshalledP3/cagi2.multianno.missenseFalse.RegionsNone.m.min0",
    "marshalledP3/cagi3.multianno.missenseFalse.RegionsNone.m.min0",
    "marshalledP3/cagi4.multianno.missenseFalse.RegionsNone.m.min0",
]

global_explainer = None  # initialized in main

In [ ]:
# ── helpers ────────────────────────────────────────────────────────────────

def load_pkl(path):
    """Load Python-2-pickled files (CAGI datasets use \r\n line endings)."""
    with open(path, 'rb') as f:
        content = f.read()
    return pickle.loads(content.replace(b'\r\n', b'\n'), encoding='latin1')


def _set_params(model, params):
    """Push a flat list of numpy arrays into a torch model's state dict."""
    params_dict = zip(model.state_dict().keys(), params)
    state_dict = OrderedDict({k: t.from_numpy(np.copy(v)) for k, v in params_dict})
    model.load_state_dict(state_dict, strict=True)


def splitData(x, y, folds, shuffle=True):
    """Stratified K-Fold split — mirrors the original standaloneFL.py."""
    X, Y = np.array(x), np.array(y)
    skf = StratifiedKFold(n_splits=folds, shuffle=shuffle)
    datasets = []
    for _, test in skf.split(X, Y):
        datasets.append((list(X[test]), list(Y[test])))
    s = sum(len(d[0]) for d in datasets)
    assert s == len(x), f"Split size mismatch: {s} != {len(x)}"
    return datasets

In [ ]:
def main(CV_FOLDS=5, use_personalized=False):
    global global_explainer

    NUM_CLIENTS = CV_FOLDS - 1

    weightPhenoPGenes, phenoPGenes = readPhenopedia("phenopediaCrohnGenes/CrohnGenes.txt")
    geneList = sorted(load_pkl(f"marshalledP3/totGeneSet.m.min{MIN_REFERENCE_NUM}"))

    # Build adjacency matrix
    adj = build_adj_from_string(
        geneList,
        string_links_path="string_db/9606.protein.links.v12.0.txt",
        string_info_path="string_db/9606.protein.info.v12.0.txt",
        threshold=700,
        cache_path="marshalledP3/adj_cache.pkl"
    )
    if adj is None or np.array_equal(adj, np.eye(len(geneList))):
        print("Falling back to Phenopedia-based adjacency")
        adj = build_adj_phenopedia(geneList, weightPhenoPGenes)

    # Load and prepare data
    DATAX, DATAY = [], []
    HX = {}
    for D in DBs:
        db = load_pkl(D)
        for sampleName, (exome, label) in db.items():
            geneDB = scanGenes(exome.items(), geneList)
            HX[sampleName] = (buildVectorGeneWise(geneDB, geneList, weightPhenoPGenes, None), label)
        X, Y = buildFeatVect(HX, list(HX.keys()))
        DATAX += X
        DATAY += Y

    datasets = splitData(DATAX, DATAY, CV_FOLDS)
    global_explainer = GlobalExplainer(geneList)

    totRes = {"sen": [], "spe": [], "pre": [], "mcc": [], "auc": [], "auprc": []}

    for i in range(CV_FOLDS):
        print(f"\n{'='*50}")
        print(f"FOLD {i+1}/{CV_FOLDS}")
        tmpTest = datasets.pop(0)
        testX, testY = tmpTest

        def client_fn(cid: str):
            return GATFLClient(cid, datasets, geneList, adj, use_personalized)

        net = GATCrohnModel(len(testX[0][0]), len(testX[0]), adj, geneList)

        strategy = fl.server.strategy.FedAvg(
            evaluate_fn=get_evaluate_fn(net, testX, testY),
            fraction_fit=1.0,
            fraction_evaluate=1.0,
            min_fit_clients=NUM_CLIENTS,
            min_evaluate_clients=NUM_CLIENTS,
            min_available_clients=NUM_CLIENTS,
        )

        tmpRes = fl.simulation.start_simulation(
            client_fn=client_fn,
            clients_ids=list(range(NUM_CLIENTS)),
            num_clients=NUM_CLIENTS,
            config=fl.server.ServerConfig(num_rounds=5),
            strategy=strategy,
        )

        datasets.append(tmpTest)

        for metric in totRes:
            totRes[metric].append(tmpRes.metrics_centralized[metric][-1][1])

        print(f"Fold {i+1} done.")

    print("\n=== FINAL RESULTS ===")
    for metric, vals in totRes.items():
        print(f"{metric}: {np.mean(vals):.4f} ± {np.std(vals):.4f}")

    global_explainer.save_report("results/gene_importance.txt")

In [ ]:
class GATFLClient(fl.client.NumPyClient):
    
    def __init__(self, cid, datasets, geneList, adj, personalized=False):
        self.cid = int(cid)
        self.X, self.Y = datasets[self.cid]
        self.geneList = geneList
        
        base_net = GATCrohnModel(
            len(self.X[0][0]), len(self.X[0]), adj, geneList
        )
        
        if personalized:
            self.net = PersonalizedGATCrohn(base_net)
            self.personalized = True
        else:
            self.net = base_net
            self.personalized = False
        
        from sources.GraphConv import NNwrapper
        self.wrapper = NNwrapper(self.net)
        self.explainer = LocalExplainer(self.net if not personalized 
                                        else self.net.backbone, geneList)
    
    def get_parameters(self, config):
        if self.personalized:
            return self.net.get_shared_params()
        return [v.cpu().numpy() for _, v in self.net.state_dict().items()]
    
    def fit(self, parameters, config):
        if self.personalized:
            self.net.set_shared_params(parameters)
        else:
            _set_params(self.net, parameters)
        
        self.wrapper.fit(
            self.X, self.Y, epochs=100, batch_size=3,
            weight_decay=1, learning_rate=1e-3, silent=True
        )
        
        if self.personalized:
            self.net.fine_tune_local(self.X, self.Y, epochs=20)
        
        # Collect local explanations
        local_imp = self.explainer.get_attention_importance(self.X)
        imp_vec = self.explainer.serialize(local_imp)
        
        # Aggregate on server side (pass via metrics)
        # Note: in real deployment, use a custom strategy for this
        
        return self.get_parameters(config={}), len(self.X), {}
    
    def evaluate(self, parameters, config):
        if self.personalized:
            self.net.set_shared_params(parameters)
        else:
            _set_params(self.net, parameters)
        
        Yp = self.wrapper.predict(self.X, batch_size=len(self.X))
        sen, spe, acc, bac, pre, mcc, auc, auprc = U.getScoresSVR(
            Yp, self.Y, threshold=None, invert=False,
            PRINT=False, CURVES=False, SAVEFIG=None
        )
        return mcc, len(self.X), {"auc": float(auc), "auprc": float(auprc)}

In [ ]:
def get_evaluate_fn(model, X, Y):
    def evaluate(server_round, parameters, config):
        _set_params(model, parameters)
        from sources.GraphConv import NNwrapper
        wrapper = NNwrapper(model)
        Yp = wrapper.predict(X, batch_size=len(X))
        sen, spe, acc, bac, pre, mcc, auc, auprc = U.getScoresSVR(
            Yp, Y, threshold=None, invert=False,
            PRINT=True, CURVES=False, SAVEFIG=None
        )
        
        # Update global explainer
        if global_explainer is not None:
            imp = model.get_gene_importance()
            if imp is not None:
                global_explainer.round_importances.append(
                    imp.cpu().numpy()
                )
        
        return mcc, {
            "sen": sen, "spe": spe, "pre": pre,
            "mcc": mcc, "auc": auc, "auprc": auprc
        }
    return evaluate

In [ ]:
if __name__ == "__main__":
    main(sys.argv)